In [ ]:
# import json
# import requests
# from dotenv import load_dotenv
# from typing import TypedDict, Annotated
# from IPython.display import Image, display

# # LangChain / Anthropic integrations
# from langchain_anthropic import ChatAnthropic
# from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
# from langchain_core.tools import tool

# # LangGraph runtime primitives
# from langgraph.graph import StateGraph, START, END
# from langgraph.graph.message import add_messages
# from langgraph.prebuilt import ToolNode

# load_dotenv()

# # ----------------------------------------------------
# # 1. Define Tool with Safe Exception Handling
# # ----------------------------------------------------
# @tool
# def get_conversion_rate(base_currency: str, target_currency: str) -> str:
#     """Retrieves the current exchange rate between two specified currencies.

#     Uses standard 3-letter ISO 4217 currency codes to fetch real-time exchange rates.
#     """
#     url = f'https://v6.exchangerate-api.com/v6/d0397a29769416a679436009/pair/{base_currency}/{target_currency}'

#     try:
#         response = requests.get(url, timeout=10)
#         response.raise_for_status()
#         data = response.json()
#         return json.dumps(data)
#     except requests.exceptions.RequestException as req_err:
#         # Return string errors so LLM can read and handle them
#         return f"Error fetching exchange rate: {str(req_err)}"


# # ----------------------------------------------------
# # 2. Define State & Initialize Model
# # ----------------------------------------------------
# class ConversionNodeState(TypedDict):
#     # 'add_messages' ensures conversation history is preserved across agent tool loops
#     messages: Annotated[list, add_messages]
#     conversion_result: str


# tools = [get_conversion_rate]

# model = ChatAnthropic(
#     model="claude-haiku-4-5", # Updated to standard available model identifier
#     max_tokens=1024
# )

# model_with_tools = model.bind_tools(
#     tools=tools, 
#     tool_choice="auto", 
#     parallel_tool_calls=False
# )

# # ----------------------------------------------------
# # 3. Define Graph Nodes & Conditional Routing
# # ----------------------------------------------------
# SYSTEM_PROMPT = SystemMessage(
#     content="You are a precise, single-purpose Currency Conversion Agent. "
#     "Your sole task is to take a financial value in one currency specified by the user. "
#     "If the user inputs something you can't understand or is ambiguous, ask for more clarity, "
#     "query an external exchange rate tool, and convert that value into the target currency."
# )

# def conversion_node(state: ConversionNodeState) -> dict:
#     messages = state["messages"]
    
#     # Prepend SystemMessage if not present
#     if not isinstance(messages[0], SystemMessage):
#         messages = [SYSTEM_PROMPT] + messages

#     # Properly call invoke to get AIMessage response
#     response = model_with_tools.invoke(messages)
    
#     return {
#         "messages": [response],
#         "conversion_result": response.content if not response.tool_calls else ""
#     }

# def should_continue(state: ConversionNodeState) -> str:
#     """Routes execution to tools node if LLM wants to call a tool, else ends."""
#     last_message = state["messages"][-1]
#     if hasattr(last_message, "tool_calls") and last_message.tool_calls:
#         return "tools"
#     return END

# # ----------------------------------------------------
# # 4. Construct LangGraph Workflow
# # ----------------------------------------------------
# workflow = StateGraph(ConversionNodeState)

# # Add Nodes
# workflow.add_node("conversion_node", conversion_node)
# workflow.add_node("tools", ToolNode(tools))

# # Add Edges
# workflow.add_edge(START, "conversion_node")

# # Conditional Edge: If tool requested -> go to "tools", else -> END
# workflow.add_conditional_edges(
#     "conversion_node",
#     should_continue,
#     {
#         "tools": "tools",
#         END: END
#     }
# )

# # Loop back from tools node to conversion_node to let Claude process tool results
# workflow.add_edge("tools", "conversion_node")

# # Compile Runnable App
# app = workflow.compile()

# # Display Mermaid PNG Graph Schema
# try:
#     display(Image(app.get_graph().draw_mermaid_png()))
# except Exception:
#     pass

# # ----------------------------------------------------
# # 5. Example Execution Run
# # ----------------------------------------------------
# if __name__ == "__main__":
#     initial_input = {
#         "messages": [HumanMessage(content="Convert 100 USD to EUR and 100 GBP to JPY.")]
#     }
    
#     final_output = app.invoke(initial_input)
#     print("\n--- Final Assistant Response ---")
#     print(final_output["messages"][-1].content)

In [ ]:
# langchain

from dataclasses import dataclass

from dotenv import load_dotenv
import json
import requests
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

@dataclass
class Context:
    user_id: str


@dataclass
class ResponseFormat:
    base_currency: float
    target_currency: float
    conversion_rate: float
    calculation: str
    location: str




@tool
def get_conversion_rate(base_currency: str, target_currency: str) -> str:
    """Retrieves the current exchange rate between two specified currencies.

    Uses standard 3-letter ISO 4217 currency codes to fetch real-time exchange rates.
    """
    url = f'https://v6.exchangerate-api.com/v6/d0397a29769416a679436009/pair/{base_currency}/{target_currency}'

    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        print(data)
        return json.dumps(data)
    except requests.exceptions.RequestException as req_err:
        # Return string errors so LLM can read and handle them
        return f"Error fetching exchange rate: {str(req_err)}"

@tool
def locate_user(runtime: ToolRuntime[Context]):
    """"return the location of the user"""
    match runtime.context.user_id:
        case "ABC123":
            return 'Vienna'
        case "XYZ456":
            return "London"
        case "HJKL":
            return "Paris"
        case _:
            return "Unknown"

checkpointer = InMemorySaver()

agent = create_agent(
    model='claude-haiku-4-5',
    tools=[get_conversion_rate, locate_user],
    system_prompt="You are a currency-conversion agent. Show full working showing the conversion steps",
    context_schema=Context,
    response_format=ResponseFormat,
    checkpointer=checkpointer
)

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke({
    "messages": [
        {"role": "user", "content": "Convert 100 USD to EUR"}
    ]},
    config=config,
    context=Context(user_id="ABC123")
    )

print(response["structured_response"])
# print(response["messages"][-1].content)




[2026-08-09 19:29:29 - anthropic._base_client:515 - DEBUG] Request options: {'method': 'post', 'url': '/v1/messages', 'headers': {'anthropic-user-profile-id': <anthropic.Omit object at 0x0000022A92370F10>}, 'files': None, 'idempotency_key': 'stainless-python-retry-21831dad-e4c4-4f05-bcc0-285b0e5536fc', 'content': None, 'json_data': {'max_tokens': 64000, 'messages': [{'role': 'user', 'content': 'Convert 100 USD to EUR'}], 'model': 'claude-haiku-4-5', 'output_config': {'format': {'type': 'json_schema', 'schema': {'type': 'object', 'title': 'ResponseFormat', 'properties': {'base_currency': {'type': 'number', 'title': 'Base Currency'}, 'target_currency': {'type': 'number', 'title': 'Target Currency'}, 'conversion_rate': {'type': 'number', 'title': 'Conversion Rate'}, 'calculation': {'type': 'string', 'title': 'Calculation'}, 'location': {'type': 'string', 'title': 'Location'}}, 'additionalProperties': False, 'required': ['base_currency', 'target_currency', 'conversion_rate', 'calculation',

{'result': 'success', 'documentation': 'https://www.exchangerate-api.com/docs', 'terms_of_use': 'https://www.exchangerate-api.com/terms', 'time_last_update_unix': 1786233601, 'time_last_update_utc': 'Sun, 09 Aug 2026 00:00:01 +0000', 'time_next_update_unix': 1786320001, 'time_next_update_utc': 'Mon, 10 Aug 2026 00:00:01 +0000', 'base_code': 'USD', 'target_code': 'EUR', 'conversion_rate': 0.8659}


[2026-08-09 19:29:34 - httpx:1025 - INFO] HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
[2026-08-09 19:29:34 - anthropic._base_client:1311 - DEBUG] HTTP Response: POST https://api.anthropic.com/v1/messages "200 OK" Headers({'date': 'Sun, 09 Aug 2026 17:29:33 GMT', 'content-type': 'application/json', 'transfer-encoding': 'chunked', 'connection': 'keep-alive', 'anthropic-ratelimit-input-tokens-limit': '10000000', 'anthropic-ratelimit-input-tokens-remaining': '9999000', 'anthropic-ratelimit-input-tokens-reset': '2026-08-09T17:29:33Z', 'anthropic-ratelimit-output-tokens-limit': '2000000', 'anthropic-ratelimit-output-tokens-remaining': '2000000', 'anthropic-ratelimit-output-tokens-reset': '2026-08-09T17:29:33Z', 'anthropic-ratelimit-requests-limit': '10000', 'anthropic-ratelimit-requests-remaining': '9999', 'anthropic-ratelimit-requests-reset': '2026-08-09T17:29:32Z', 'anthropic-ratelimit-tokens-limit': '12000000', 'anthropic-ratelimit-tokens-remaining': '119990

ResponseFormat(base_currency=100.0, target_currency=86.59, conversion_rate=0.8659, calculation='100 USD × 0.8659 = 86.59 EUR', location='Vienna')


In [ ]:
# from dataclasses import dataclass

# from dotenv import load_dotenv
# import json
# import requests
# from langchain.agents import create_agent
# from langchain.tools import tool, ToolRuntime
# from langgraph.checkpoint.memory import InMemorySaver

# from sentence_transformers import SentenceTransformer

# # model = SentenceTransformer("BAAI/bge-m3")
# model = SentenceTransformer("all-MiniLM-L6-v2")

# # The sentences to encode
# texts = [
#     "I love apples.",
#     "I enjoy oranges.",
#     "I think pears taste very good.",
#     "I hate bananas.",
#     "I dislike rasberries.",
#     "I despice mangoes.",
#     "I love linux",
#     "I hate windows."
# ]

# query1 = model.encode("What fruits does the person like?")
# query2 = model.encode("What fruits does the person hate?")

# vector_store = model.encode(texts)

# print(model.similarity(query1, vector_store))
# print(model.similarity(query2, vector_store))










[2026-08-09 22:32:53 - httpx:1025 - INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
[2026-08-09 22:32:53 - httpx:1025 - INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
[2026-08-09 22:32:53 - httpx:1025 - INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
[2026-08-09 22:32:53 - httpx:1025 - INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"
[2026-08-09 22:32:54 - httpx:1025 - INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_

tensor([[0.5061, 0.5058, 0.5724, 0.4510, 0.4654, 0.4457, 0.1150, 0.1212]])
tensor([[0.4721, 0.4682, 0.4883, 0.6207, 0.5879, 0.4305, 0.1198, 0.3117]])


In [14]:

from dotenv import load_dotenv
from typing import TypedDict, Annotated, Literal
from pydantic import BaseModel, Field

from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document
import uuid
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()

llm = init_chat_model('claude-haiku-4-5')


KNOWLEDGE = [
    "NeuralNine is a YouTube channel focused on programming, AI, and software engineering tutorials.",
    "LangGraph is a library for building stateful, multi-agent applications on top of LangChain.",
    "A StateGraph in LangGraph defines nodes and edges that operate on a shared typed state.",
    "Checkpointers Like InMemorySaver let LangGraph persist conversation state across invocations using a thread_id.",
    "RAG (Retrieval-Augmented Generation) combines a retriever over a knowledge base with an LLM to ground answers in source documents."
]

vector_store = InMemoryVectorStore(HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"))
vector_store.add_documents([Document(page_content=text) for text in KNOWLEDGE])


class IntentClassifier(BaseModel):
    message_intent: Literal["chat", "knowledge", "code"] = Field(..., description="Classify whether the uset wants to just chat, ask for knowledge or change code in the project.")

class State(TypedDict):
    messages: Annotated[list, add_messages]
    message_intent: str | None



def classify_intent(state: State):
    structured_llm = llm.with_structured_output(IntentClassifier)
    result = structured_llm.invoke([
    {"role": "system", "content": "Determine / classify whether the user wants to chat, retrieve knowledge or change code. Output either: chat, knowledge or code"},
    {"role": "user", "content": state["messages"][-1].content}
    ])

    print("user intent ->", result.message_intent)
    return {"message_intent": result.message_intent}


def prompt_llm_chat(state: State):
    messages = [{"role": "system", "content": "You are a talkative chatbot for fun. Be nice"}] + state["messages"]

    response = llm.invoke(messages)

    return {"messages": [{"role": "assistant", "content": response.content}]}

def prompt_llm_rag(state: State):
    query = state["messages"][-1].content
    documents = vector_store.similarity_search(query, k=3)

    # print("documents -> ", documents)

    context = "\n".join(f"{doc.page_content}" for doc in documents)

    print(state["messages"])
    messages = [
    {'role': 'system', 'content': f'You are a RAG agent. Answer the user using only the context below. If the answer is not in it only say you don\'t know.\n\nContext:\n{context}'}
    ] + state['messages']

    response = llm.invoke(messages)

    return {"messages": [{"role": "assistant", "content": response.content}]}

def prompt_llm_code(state: State):
    messages = [{"role": "system", "content": "No matter what the user say, always say: I am the CODING agent. Do not say anything else"}] + state["messages"]

    response = llm.invoke(messages)

    return {"messages": [{"role": "assistant", "content": response.content}]}


graph_builder = StateGraph(State)

graph_builder.add_node("classifier", classify_intent)
graph_builder.add_node("chat_agent", prompt_llm_chat)
graph_builder.add_node("rag_agent", prompt_llm_rag)
graph_builder.add_node("coding_agent", prompt_llm_code)

graph_builder.add_edge(START, "classifier")
graph_builder.add_conditional_edges("classifier", lambda state: state["message_intent"],  
                                    {"chat": "chat_agent", "knowledge": "rag_agent", "code": "coding_agent"})
graph_builder.add_edge("chat_agent", END)
graph_builder.add_edge("rag_agent", END)
graph_builder.add_edge("coding_agent", END)

checkpointer = InMemorySaver()
graph = graph_builder.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": uuid.uuid4()}}

graph.get_graph().draw_mermaid_png(output_file_path="graph.png")

while True:
    user_message = input("Enter message:")
    result = graph.invoke({"messages": [{"role": "user", "content": user_message}]}, config=config)

    print(result["messages"][-1].content)






[2026-08-10 01:11:49 - httpx:1025 - INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
[2026-08-10 01:11:49 - httpx:1025 - INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
[2026-08-10 01:11:49 - httpx:1025 - INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
[2026-08-10 01:11:49 - httpx:1025 - INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"
[2026-08-10 01:11:50 - httpx:1025 - INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_

user intent -> chat


[2026-08-10 01:12:09 - httpx:1025 - INFO] HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
[2026-08-10 01:12:09 - anthropic._base_client:1311 - DEBUG] HTTP Response: POST https://api.anthropic.com/v1/messages "200 OK" Headers({'date': 'Sun, 09 Aug 2026 23:12:09 GMT', 'content-type': 'application/json', 'transfer-encoding': 'chunked', 'connection': 'keep-alive', 'anthropic-ratelimit-input-tokens-limit': '10000000', 'anthropic-ratelimit-input-tokens-remaining': '10000000', 'anthropic-ratelimit-input-tokens-reset': '2026-08-09T23:12:08Z', 'anthropic-ratelimit-output-tokens-limit': '2000000', 'anthropic-ratelimit-output-tokens-remaining': '2000000', 'anthropic-ratelimit-output-tokens-reset': '2026-08-09T23:12:09Z', 'anthropic-ratelimit-requests-limit': '10000', 'anthropic-ratelimit-requests-remaining': '9999', 'anthropic-ratelimit-requests-reset': '2026-08-09T23:12:08Z', 'anthropic-ratelimit-tokens-limit': '12000000', 'anthropic-ratelimit-tokens-remaining': '12000

Hey! 👋 How's it going? I'm so glad you stopped by! What's on your mind today? Whether you want to chat about something interesting, need help with something, or just feel like hanging out and talking – I'm totally here for it! 

What brings you my way? 😊


[2026-08-10 01:12:13 - anthropic._base_client:515 - DEBUG] Request options: {'method': 'post', 'url': '/v1/messages', 'headers': {'anthropic-user-profile-id': <anthropic.Omit object at 0x000001B43A1DF890>}, 'files': None, 'idempotency_key': 'stainless-python-retry-71624d0e-98a1-44b6-8c8b-87affa6c81c3', 'content': None, 'json_data': {'max_tokens': 64000, 'messages': [{'role': 'user', 'content': ''}], 'model': 'claude-haiku-4-5', 'system': 'Determine / classify whether the user wants to chat, retrieve knowledge or change code. Output either: chat, knowledge or code', 'tool_choice': {'type': 'tool', 'name': 'IntentClassifier'}, 'tools': [{'name': 'IntentClassifier', 'input_schema': {'properties': {'message_intent': {'description': 'Classify whether the uset wants to just chat, ask for knowledge or change code in the project.', 'enum': ['chat', 'knowledge', 'code'], 'type': 'string'}}, 'required': ['message_intent'], 'type': 'object'}, 'description': ''}]}}
[2026-08-10 01:12:13 - anthropic

BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.0: user messages must have non-empty content'}, 'request_id': 'req_011Cdt2B8yzVJ8TBFqfW6QAp'}